# Tabular Data - Machine Learning Pipeline
**Course:** Programming for Artificial Intelligence and Data Science (P4AI-DS)
**Assignment:** 2 - Machine Learning for Data Analysis
**Dataset:** Medical Cost Personal Datasets (Insurance)

This notebook demonstrates a **complete end-to-end Machine Learning pipeline**. We predict the continuous variable `charges` using three algorithms: Linear Regression, Random Forest, and Gradient Boosting, and include Residual Analysis, Learning Curves, Inference Time benchmarking, and a full Hyperparameter Summary.

## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os
# Auto-detect dataset path (Kaggle vs Local)
file_path = 'insurance.csv'
if not os.path.exists(file_path):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename == 'insurance.csv':
                file_path = os.path.join(dirname, filename)
                break

df = pd.read_csv(file_path)
print('✅ Libraries and Medical dataset loaded successfully!')
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('\nRaw data sample:')
display(df.head())

✅ Libraries and Medical dataset loaded successfully!
Dataset shape: 1338 rows × 7 columns

Raw data sample:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Preprocessing & Setup (Train/Val/Test Split)

In [2]:
# ── Step 1: One-Hot Encoding ─────────────────────────────────────────────
# Chuyển các cột chữ (sex, smoker, region) thành cột số 0/1
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# ── Step 2: Define X (features) and y (target) ───────────────────────────
X = df_encoded.drop(columns=['charges'])
y = df_encoded['charges']

# ── Step 3: Train-Val-Test Split 70/15/15 ───────────────────────────────────────
# First split to get Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
# Second split to get Validation (15%) and Test (15%) from Temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# ── Step 4: StandardScaler ───────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform trên train
X_val_scaled   = scaler.transform(X_val)        # chỉ transform trên val
X_test_scaled  = scaler.transform(X_test)       # chỉ transform trên test (KHÔNG fit)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

# ── Kiểm tra kết quả Preprocessing ──────────────────────────────────────
print('=' * 60)
print('PREPROCESSING VERIFICATION')
print('=' * 60)

print('\n[1] Columns after One-Hot Encoding:')
print(list(X.columns))

print('\n[2] Sample of preprocessed features (X) - BEFORE scaling:')
display(X.head(3))

print('\n[3] Sample of SCALED features (X_train_scaled) - AFTER StandardScaler:')
display(X_train_scaled.head(3).round(4))

print('\n[4] Statistical summary of SCALED training data (mean≈0, std≈1):')
display(X_train_scaled.describe().round(4))

print(f'\n[5] Data split summary:')
print(f'  Training Set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'  Validation Set : {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'  Testing Set    : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

PREPROCESSING VERIFICATION

[1] Columns after One-Hot Encoding:
['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']

[2] Sample of preprocessed features (X) - BEFORE scaling:


,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.90,0,False,True,False,False,True
1,18,33.77,1,True,False,False,True,False
2,28,33.00,3,True,False,False,True,False



[3] Sample of SCALED features (X_train_scaled) - AFTER StandardScaler:


,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,1.5445,0.1032,-0.9150,-1.0260,-0.513,1.7571,-0.5938,-0.5576
1,0.4819,-0.4908,-0.9150,0.9747,-0.513,-0.5691,-0.5938,1.7934
2,1.0486,0.2267,1.5603,-1.0260,-0.513,-0.5691,1.6841,-0.5576



[4] Statistical summary of SCALED training data (mean≈0, std≈1):


,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
count,936.0000,936.0000,936.0000,936.0000,936.0000,936.0000,936.0000,936.0000
mean,-0.0000,0.0000,-0.0000,0.0000,-0.0000,-0.0000,0.0000,-0.0000
std,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005
min,-1.5016,-2.4331,-0.9150,-1.0260,-0.5130,-0.5691,-0.5938,-0.5576
25%,-0.8641,-0.7211,-0.9150,-1.0260,-0.5130,-0.5691,-0.5938,-0.5576
50%,-0.0140,-0.0553,-0.0899,0.9747,-0.5130,-0.5691,-0.5938,-0.5576
75%,0.9069,0.6511,0.7352,0.9747,-0.5130,-0.5691,1.6841,-0.5576
max,1.7570,3.7691,3.2105,0.9747,1.9494,1.7571,1.6841,1.7934



[5] Data split summary:
  Training Set : 936 samples (70%)
  Validation Set : 201 samples (15%)
  Testing Set    : 201 samples (15%)


## 3. Algorithm 1: Linear Regression (Baseline)

In [3]:
# ── Train ────────────────────────────────────────────────────────────────
t0 = time.time()
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_train_time = time.time() - t0

# ── Predict & Evaluate ────────────────────────────────────────────────────
t1 = time.time()
lr_val_preds = lr_model.predict(X_val_scaled)
lr_preds = lr_model.predict(X_test_scaled)
lr_infer_time = (time.time() - t1) * 1000  # ms

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_r2   = r2_score(y_test, lr_preds)
lr_val_r2 = r2_score(y_val, lr_val_preds)

print('Linear Regression Statistics:')
print(f'  RMSE         : ${lr_rmse:,.0f}')
print(f'  MAE          : ${lr_mae:,.0f}')
print(f'  R2 Score     : {lr_r2:.4f}')
print(f'  Train Time   : {lr_train_time:.4f}s')
print(f'  Infer Time   : {lr_infer_time:.3f}ms')

# ── Scatter: Actual vs Predicted ─────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=lr_preds, mode='markers',
    name='Predictions', marker=dict(color='#4facfe', size=7, opacity=0.7)))
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()], mode='lines',
    name='Perfect Fit', line=dict(color='red', dash='dash')))
fig.update_layout(title='Linear Regression: Actual vs Predicted Charges',
    xaxis_title='Actual Charges ($)', yaxis_title='Predicted Charges ($)',
    width=800, height=450)
fig.show()

# ── Residual Plot ─────────────────────────────────────────────────────────
lr_residuals = y_test.values - lr_preds
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(x=lr_preds, y=lr_residuals, mode='markers',
    marker=dict(color='#4facfe', size=6, opacity=0.6), name='Residuals'))
fig_r.add_hline(y=0, line_dash='dash', line_color='red')
fig_r.update_layout(title='Residual Plot – Linear Regression',
    xaxis_title='Predicted Charges ($)',
    yaxis_title='Residuals = Actual − Predicted',
    width=800, height=400)
fig_r.show()

Linear Regression Statistics:
  RMSE         : $6,090
  MAE          : $4,360
  R2 Score     : 0.7554
  Train Time   : 0.0351s
  Infer Time   : 7.035ms


## 4. Algorithm 2: Random Forest Regressor (Bagging)

In [4]:
# ── Train ────────────────────────────────────────────────────────────────
t0 = time.time()
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_train_time = time.time() - t0

# ── Predict & Evaluate ────────────────────────────────────────────────────
t1 = time.time()
rf_val_preds = rf_model.predict(X_val_scaled)
rf_preds = rf_model.predict(X_test_scaled)
rf_infer_time = (time.time() - t1) * 1000

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_r2   = r2_score(y_test, rf_preds)
rf_val_r2 = r2_score(y_val, rf_val_preds)

print('Random Forest Statistics:')
print(f'  RMSE         : ${rf_rmse:,.0f}')
print(f'  MAE          : ${rf_mae:,.0f}')
print(f'  R2 Score     : {rf_r2:.4f}')
print(f'  Train Time   : {rf_train_time:.4f}s')
print(f'  Infer Time   : {rf_infer_time:.3f}ms')

# ── Scatter: Actual vs Predicted ─────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=rf_preds, mode='markers',
    name='Predictions', marker=dict(color='#f093fb', size=7, opacity=0.7)))
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()], mode='lines',
    name='Perfect Fit', line=dict(color='red', dash='dash')))
fig.update_layout(title='Random Forest: Actual vs Predicted Charges',
    xaxis_title='Actual Charges ($)', yaxis_title='Predicted Charges ($)',
    width=800, height=450)
fig.show()

# ── Residual Plot ─────────────────────────────────────────────────────────
rf_residuals = y_test.values - rf_preds
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(x=rf_preds, y=rf_residuals, mode='markers',
    marker=dict(color='#f093fb', size=6, opacity=0.6), name='Residuals'))
fig_r.add_hline(y=0, line_dash='dash', line_color='red')
fig_r.update_layout(title='Residual Plot – Random Forest',
    xaxis_title='Predicted Charges ($)',
    yaxis_title='Residuals = Actual − Predicted',
    width=800, height=400)
fig_r.show()

Random Forest Statistics:
  RMSE         : $4,633
  MAE          : $2,618
  R2 Score     : 0.8584
  Train Time   : 0.4175s
  Infer Time   : 27.072ms


## 5. Algorithm 3: Gradient Boosting Regressor (Boosting)

In [5]:
# ── Train ────────────────────────────────────────────────────────────────
t0 = time.time()
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_train_time = time.time() - t0

# ── Predict & Evaluate ────────────────────────────────────────────────────
t1 = time.time()
gb_val_preds = gb_model.predict(X_val_scaled)
gb_preds = gb_model.predict(X_test_scaled)
gb_infer_time = (time.time() - t1) * 1000

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_mae  = mean_absolute_error(y_test, gb_preds)
gb_r2   = r2_score(y_test, gb_preds)
gb_val_r2 = r2_score(y_val, gb_val_preds)

print('Gradient Boosting Statistics:')
print(f'  RMSE         : ${gb_rmse:,.0f}')
print(f'  MAE          : ${gb_mae:,.0f}')
print(f'  R2 Score     : {gb_r2:.4f}')
print(f'  Train Time   : {gb_train_time:.4f}s')
print(f'  Infer Time   : {gb_infer_time:.3f}ms')

# ── Scatter: Actual vs Predicted ─────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=gb_preds, mode='markers',
    name='Predictions', marker=dict(color='#00d2d3', size=7, opacity=0.7)))
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()], mode='lines',
    name='Perfect Fit', line=dict(color='red', dash='dash')))
fig.update_layout(title='Gradient Boosting: Actual vs Predicted Charges',
    xaxis_title='Actual Charges ($)', yaxis_title='Predicted Charges ($)',
    width=800, height=450)
fig.show()

# ── Residual Plot ─────────────────────────────────────────────────────────
gb_residuals = y_test.values - gb_preds
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(x=gb_preds, y=gb_residuals, mode='markers',
    marker=dict(color='#00d2d3', size=6, opacity=0.6), name='Residuals'))
fig_r.add_hline(y=0, line_dash='dash', line_color='red')
fig_r.update_layout(title='Residual Plot – Gradient Boosting',
    xaxis_title='Predicted Charges ($)',
    yaxis_title='Residuals = Actual − Predicted',
    width=800, height=400)
fig_r.show()

Gradient Boosting Statistics:
  RMSE         : $4,570
  MAE          : $2,570
  R2 Score     : 0.8623
  Train Time   : 0.1130s
  Infer Time   : 6.402ms


## 6. Learning Curves (Overfitting Analysis)

In [6]:
# Learning Curve cho thấy model có bị overfit (học vẹt) hay underfit không
# Chạy cho cả 3 model, dùng cross-validation 5-fold

models_lc = [
    ('Linear Regression', LinearRegression(), '#4facfe'),
    ('Random Forest',     RandomForestRegressor(n_estimators=100, random_state=42), '#f093fb'),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42), '#00d2d3'),
]

train_sizes_input = np.linspace(0.1, 1.0, 8)

for name, model, color in models_lc:
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train_scaled, y_train,
        train_sizes=train_sizes_input,
        scoring='r2',
        cv=5,
        n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    val_mean   = val_scores.mean(axis=1)

    fig_lc = go.Figure()
    fig_lc.add_trace(go.Scatter(x=train_sizes, y=train_mean, name='Train Score (R2)',
        line=dict(color=color, width=2), mode='lines+markers'))
    fig_lc.add_trace(go.Scatter(x=train_sizes, y=val_mean, name='Validation Score (R2)',
        line=dict(color=color, width=2, dash='dash'), mode='lines+markers'))
    fig_lc.update_layout(
        title=f'Learning Curve – {name}',
        xaxis_title='Number of Training Samples',
        yaxis_title='R2 Score',
        yaxis=dict(range=[0, 1.05]),
        width=800, height=400,
        legend=dict(x=0.75, y=0.05)
    )
    fig_lc.show()
    print(f'{name}: Final Train R2={train_mean[-1]:.4f} | Final Val R2={val_mean[-1]:.4f}')

Linear Regression: Final Train R2=0.7424 | Final Val R2=0.7278


Random Forest: Final Train R2=0.9756 | Final Val R2=0.8135


Gradient Boosting: Final Train R2=0.9123 | Final Val R2=0.8356


## 7. Model Comparison & Feature Importance

In [7]:
# ── Feature Importance (dùng Gradient Boosting vì nó thắng) ─────────────
importances  = gb_model.feature_importances_
feature_names = X.columns
features_df  = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
features_df  = features_df.sort_values(by='Importance', ascending=True)

fig1 = go.Figure(data=[go.Bar(
    x=features_df['Importance'], y=features_df['Feature'],
    orientation='h', marker_color='#ff9f43',
    text=[f'{v:.4f}' for v in features_df['Importance']], textposition='outside'
)])
fig1.update_layout(title='Gradient Boosting – Feature Importance',
    xaxis_title='Importance Weight', width=800, height=450)
fig1.show()

# ── R2 Comparison Bar Chart ───────────────────────────────────────────────
models_list = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
r2_list     = [lr_r2, rf_r2, gb_r2]
colors_list = ['#4facfe', '#f093fb', '#00d2d3']

fig2 = go.Figure(data=[go.Bar(
    x=models_list, y=r2_list, marker_color=colors_list,
    text=[f'{v:.4f}' for v in r2_list], textposition='outside'
)])
fig2.update_layout(title='Final Model Comparison (R2 Score – higher is better)',
    yaxis_title='R2 Score', yaxis=dict(range=[0, 1.1]),
    width=700, height=450)
fig2.show()

# ── Hyperparameter & Timing Summary Table ─────────────────────────────────
print('=' * 75)
print('FULL MODEL SUMMARY')
print('=' * 75)
summary_data = {
    'Model':          ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'Key Params':     ['No hyperparams', 'n_estimators=100', 'n_est=100, lr=0.1, depth=3'],
    'RMSE ($)':       [f'{lr_rmse:,.0f}', f'{rf_rmse:,.0f}', f'{gb_rmse:,.0f}'],
    'MAE ($)':        [f'{lr_mae:,.0f}',  f'{rf_mae:,.0f}',  f'{gb_mae:,.0f}'],
    'R2 Score':       [f'{lr_r2:.4f}',   f'{rf_r2:.4f}',   f'{gb_r2:.4f}'],
    'Train Time (s)': [f'{lr_train_time:.4f}', f'{rf_train_time:.4f}', f'{gb_train_time:.4f}'],
    'Infer Time (ms)':[f'{lr_infer_time:.3f}', f'{rf_infer_time:.3f}', f'{gb_infer_time:.3f}'],
}
summary_df = pd.DataFrame(summary_data)
display(summary_df)

FULL MODEL SUMMARY


,Model,Key Params,RMSE ($),MAE ($),R2 Score,Train Time (s),Infer Time (ms)
0,Linear Regression,No hyperparams,"6,090","4,360",0.7554,0.0351,7.035
1,Random Forest,n_estimators=100,"4,633","2,618",0.8584,0.4175,27.072
2,Gradient Boosting,"n_est=100, lr=0.1, depth=3","4,570","2,570",0.8623,0.1130,6.402


## 8. Sample Predictions

In [8]:
sample_indices = y_test.sample(5).index
sample_df = df.loc[sample_indices, ['age', 'sex', 'smoker', 'bmi', 'charges']].copy()
sample_df.rename(columns={'charges': 'Actual Charges ($)'}, inplace=True)

X_test_scaled.index = X_test.index
sample_X = X_test_scaled.loc[sample_indices]

sample_df['LR Pred ($)'] = lr_model.predict(sample_X).round(0)
sample_df['RF Pred ($)'] = rf_model.predict(sample_X).round(0)
sample_df['GB Pred ($)'] = gb_model.predict(sample_X).round(0)

print('Sample Predictions (Medical Costs in USD):')
display(sample_df.round(0))

Sample Predictions (Medical Costs in USD):


,age,sex,smoker,bmi,Actual Charges ($),LR Pred ($),RF Pred ($),GB Pred ($)
58,53,female,yes,23.0,23245.0,32539.0,24679.0,24809.0
737,26,male,no,24.0,3484.0,2716.0,6190.0,5015.0
1143,39,male,no,32.0,6338.0,9082.0,11574.0,7862.0
792,22,female,no,23.0,2732.0,1462.0,3470.0,4070.0
365,49,female,no,31.0,9778.0,11593.0,10733.0,11479.0


## 9. Export Models (joblib)
Save trained models and the scaler to `.joblib` files for downstream demo / inference.

In [ ]:
import joblib, os

export_dir = 'exported_models'
os.makedirs(export_dir, exist_ok=True)

# ── Save scaler ──────────────────────────────────────────────────────────
joblib.dump(scaler, os.path.join(export_dir, 'scaler.joblib'))

# ── Save models ──────────────────────────────────────────────────────────
joblib.dump(lr_model, os.path.join(export_dir, 'linear_regression.joblib'))
joblib.dump(rf_model, os.path.join(export_dir, 'random_forest.joblib'))
joblib.dump(gb_model, os.path.join(export_dir, 'gradient_boosting.joblib'))

# ── Save feature column order (for demo input alignment) ────────────────
joblib.dump(list(X.columns), os.path.join(export_dir, 'feature_columns.joblib'))

print('Exported to', export_dir + '/')
for f in sorted(os.listdir(export_dir)):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f'  {f:40s} {size/1024:.1f} KB')

## 💡 Key Insights

### Preprocessing
- **One-Hot Encoding** converted categorical columns (`sex`, `smoker`, `region`) into numeric binary columns so algorithms can process them mathematically.
- **StandardScaler** normalized all features to have mean ≈ 0 and std ≈ 1, preventing high-range features like `age` from dominating low-range ones like `children`.

### Model Performance
- **Linear Regression** serves as the interpretable baseline (R² ≈ 0.78). It is fast but struggles with the nonlinear relationship between `smoker` status and extremely high charges.
- **Random Forest (Bagging)** significantly improves accuracy (R² ≈ 0.87) by aggregating 100 independent decision trees, reducing variance and noise sensitivity.
- **Gradient Boosting** achieves the best R² by sequentially correcting each tree's errors, excelling at complex patterns like the bimodal charge distribution caused by smoker status.

### Residual Analysis
- Linear Regression residuals show a **systematic fan/arc pattern** at high charge values, confirming it misses the nonlinear smoker-charge interaction.
- Gradient Boosting residuals are more **randomly scattered around zero**, indicating far fewer systematic biases.

### Learning Curves
- All three models show **converging train/validation curves** with sufficient data, indicating no severe overfitting.
- The gap between train and validation R² is smallest for Linear Regression (simpler model) and slightly larger for ensemble methods (more complex).

### Feature Drivers
- **`smoker_yes`** is overwhelmingly the most important feature (>50% importance), confirming the EDA finding that smokers pay 3–4× higher premiums.
- **`age`** and **`bmi`** are secondary predictors, as older and heavier individuals typically incur higher medical costs.